# M2: CLV 수준·구성·가격 좌표 rho=0.10 귀속 검정 (Dunnhumby, seed 42)

직전 구조와 `beta=0.25`는 그대로 유지하고 전체 보조강도만 `rho=0.10`으로 높입니다. 실제 CLV arm과 사용자 이진 degree 10분위 안에서 N·V·전체 CLV 묶음을 함께 순열한 대조군을 동일 초기화·동일 음성표본으로 학습합니다. 최종 test와 holdout은 구성하지 않습니다.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!rm -rf /content/clv-m2-lightgcn-runner
!git clone --branch feat/m2-joint-nv-lightgcn https://github.com/jung-un/clv-m2-lightgcn-runner.git /content/clv-m2-lightgcn-runner
%cd /content/clv-m2-lightgcn-runner
!git checkout 2224a58
!git rev-parse HEAD

In [ ]:
import json
import torch
from lightgcn_clv_constrained_economic_embedding import (
    configure_rho10_attribution_run,
    preflight_summary,
    run_constrained_economic_screen,
)

assert torch.cuda.is_available(), '런타임 유형에서 GPU를 선택하세요.'
cfg = configure_rho10_attribution_run()
print(json.dumps(preflight_summary(cfg), ensure_ascii=False, indent=2))

In [ ]:
result_df = run_constrained_economic_screen(cfg)

In [ ]:
from IPython.display import display
import pandas as pd

print('1) 절대지표: M1, rho=0, 실제 CLV, degree-matched shuffle, ID-only, 관계-only, 가격-only')
display(result_df)
print('2) 대조군별 전체 성과 비교')
display(pd.DataFrame(result_df.attrs['comparison']))
print('3) M1 대비 실제 CLV Top-10 변경')
display(pd.DataFrame(result_df.attrs['top10_overlap']))
print('4) degree-matched shuffle 대비 실제 CLV Top-10 변경')
display(pd.DataFrame(result_df.attrs['attribution_overlap']))
print('5) 실제 점수 영향력')
display(pd.DataFrame(result_df.attrs['score_diagnostics']))
print('6) 사전 판정 규칙 결과')
print(json.dumps(result_df.attrs['screening_reading'], ensure_ascii=False, indent=2))
print('7) 저장 파일')
print(json.dumps(result_df.attrs['result_paths'], ensure_ascii=False, indent=2))